In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import time
import numpy as np
from scipy.spatial.transform import Rotation

import jax.numpy as jnp
from jax.scipy.spatial.transform import Rotation as jaxRotation

import matplotlib.pyplot as plt

from helicopter.vision.tracking import TrianglePointMatcher, ICP

In [ ]:
# measured_points = np.array([[      1.768,  -0.0079125,    -0.43461],
#        [     1.7783,    0.021053,    -0.44639],
#        [     1.7688,  0.00070465,    -0.44736],
#        [     1.7945,     0.11468,    -0.44259],
#        [     1.7672,   -0.043063,    -0.43921],
#        [     1.7704,   -0.026978,    -0.43964],
#        [     1.7892,     0.12382,    -0.45662]])

In [ ]:
# from 04-15
measured_points = np.array([[      1.745,    0.042812,    -0.44773],
       [     1.7255,   -0.076224,    -0.44818],
       [     1.7187,   -0.096845,    -0.45425],
       [     1.7275,   -0.069065,    -0.46149],
       [     1.7251,   -0.047819,    -0.45617],
       [     1.7298,   -0.053119,    -0.44418],
       [     1.7265,   -0.039569,    -0.45025],
       [       1.72,    -0.09838,    -0.43537]])

In [ ]:
table_space_measured = np.array([[     0.2168,     0.20765,    0.086348],
       [    0.33735,     0.20337,    0.085895],
       [    0.35867,     0.19921,    0.079828],
       [       0.33,     0.20443,    0.072586],
       [    0.30923,     0.19934,    0.077905],
       [    0.31389,     0.20466,    0.089898],
       [    0.30086,     0.19975,    0.083824],
       [    0.36002,     0.20071,    0.098709]])

In [ ]:
reference_points = np.load('/home/ray/projects/helicopter/assets/point_clouds/green_syma.npy')

In [ ]:
def plot_3d(points, window, marker_size=15, c=None, show_indices=True, center=False, other_points=None):
    plt.close('all')

    fig = plt.figure(figsize=(8, 8))
    ax = fig.add_subplot(projection='3d')

    if center:
        centered_points = points - np.mean(points, axis=0)
        x = centered_points[:, 0]
        y = centered_points[:, 1]
        z = centered_points[:, 2]
    else:
        x = points[:, 0]
        y = points[:, 1]
        z = points[:, 2]

    if c is None:
        c = y

    ax.scatter(x, y, z, c=c, cmap='plasma', s=marker_size, depthshade=True)

    if show_indices:
        for _i in range(len(points)):
            ax.text(x[_i], y[_i], z[_i], str(_i), size=8, color='black', zorder=10)

    if other_points is not None:
        x = other_points[:, 0]
        y = other_points[:, 1]
        z = other_points[:, 2]

        ax.scatter(x, y, z, c='g', s=marker_size, depthshade=True)

    ax.set_xlim(-window, window)
    ax.set_ylim(-window, window)
    ax.set_zlim(-window, window)

    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')

    plt.show()

In [ ]:
matcher = TrianglePointMatcher(n=5000, k=100, inlier_threshold=0.005)
r, t = matcher.get_alignment(sample_points=table_space_measured)

In [ ]:
r.as_rotvec(degrees=True)

In [ ]:
t

In [ ]:
%matplotlib ipympl

plot_3d(r.apply(reference_points), other_points=(table_space_measured - t), window=0.1)

In [ ]:
icp = ICP(distance_threshold=1e-1, etol=0.003, max_iterations=10)

In [ ]:
def pad_points(points):
    max_size = len(reference_points)
    current_size = points.shape[0]

    pad_length = max_size - current_size

    pad_array = np.zeros((pad_length, 3), dtype=points.dtype)
    padded_points = np.vstack([points, pad_array])

    valid_input_mask = np.zeros(max_size, dtype=bool)
    valid_input_mask[:current_size] = True

    return padded_points, valid_input_mask

In [ ]:
q_jax = jaxRotation.from_quat(jnp.array(r.as_quat(canonical=True)))
padded, valid_mask = pad_points(table_space_measured)
start = time.perf_counter()
_, _, q_new, t_new = icp.iterate(q_old=q_jax, t_old=t, sample_points=padded, reference_points=reference_points, valid_input_mask=valid_mask)
end = time.perf_counter()

In [ ]:
end - start

In [ ]:
q_np = Rotation.from_quat(np.array(q_jax.as_quat(canonical=True)))

In [ ]:
q_np.as_rotvec(degrees=True)

In [ ]:
np.asarray(t_new)

In [ ]:
%matplotlib ipympl
plot_3d(q_new.apply(reference_points), other_points=(table_space_measured - np.asarray(t_new)), window=0.1)

In [ ]:
%matplotlib ipympl
plot_3d(table_space_measured, center=True, window=0.1)

In [ ]:
measured_points[:, 1:]

In [ ]:
measured_points_depth_removed = measured_points[:, 1:]
for i, point in enumerate(measured_points_depth_removed):
    print(i)
    norm = np.linalg.norm(measured_points_depth_removed - point, axis=1)
    print(norm)
    print(' ')

In [ ]:
from helicopter.utils import PointQueue

class DummyPointHandler:
    def __init__(self,
                 max_queue_size: int = 50) -> None:
        self.init_points : dict[int, PointQueue] = {}
        self.maxlen = max_queue_size

        self._next_id : int = 0

    @property
    def next_id(self) -> int:
        tmp = self._next_id
        self._next_id += 1
        return tmp

    @property
    def init_points_coords(self):
        point_list = [pq.mean() for pq in self.init_points.values()]
        points = np.array(point_list)

        return points

    def add_point(self, _point: np.ndarray) -> None:
        pq = PointQueue(self.maxlen)
        pq.enqueue(_point)
        self.init_points[self.next_id] = pq

    def register_points(self, points: np.ndarray):
        for _point in points:
            if len(self.init_points) < 1:
                self.add_point(_point)
            else:
                # Depth noise is isolated to first dim, so just check for Y and Z
                # I think this is fine if the tolerance is tight enough
                _norm = np.linalg.norm(self.init_points_coords[:, 1:] - _point[1:], axis=1)
                closest_point = np.argmin(_norm)
                if _norm[closest_point] > 0.01:
                    self.add_point(_point)
                else:
                    self.init_points[closest_point].enqueue(_point)

In [ ]:
ph = DummyPointHandler()

In [ ]:
ph.register_points(measured_points)

In [ ]:
ph.init_points_coords

In [ ]:
points_depth_removed = ph.init_points_coords[:, 1:]
for i, point in enumerate(points_depth_removed):
    print(i)
    norm = np.linalg.norm(points_depth_removed - point, axis=1)
    print(norm)
    print(' ')